In [1]:
import sys
sys.path.append("../../")

from retrieval.product_retriever import search_product_hybrid, metadata_extractor
from retrieval.reranker import rerank
import pandas as pd

from pathlib import Path
from html import escape
from IPython.display import HTML, display

from evaluation.eval import product_load_tests, evaluate_retrieval, context_relevance
from evaluation.classes import ContextRelevanceWrapper
from evaluation.reporting import (
    build_eval_question_report,
    build_retrieval_doc_rows,
    display_eval_question_report,
    export_retrieval_report_to_markdown,
    get_node_text,
)

/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/ragas/_analytics.py:86: ResourceWarning: unclosed file <_io.TextIOWrapper name='/Users/ayeustsihneyeu/Library/Application Support/ragas/uuid.json' mode='r' encoding='UTF-8'>
  user_id = json.load(open(uuid_filepath))["userid"]


In [2]:
tests = product_load_tests()

In [3]:
def format_metadata_filters(filters):
    if filters is None:
        return "none"

    filter_items = getattr(filters, "filters", None) or []
    if not filter_items:
        return "none"

    parts = []
    for item in filter_items:
        nested_filters = getattr(item, "filters", None)
        if nested_filters is not None:
            nested_text = format_metadata_filters(item)
            condition = str(getattr(item, "condition", "or")).upper()
            parts.append(f"({nested_text}) [{condition}]")
            continue

        parts.append(
            f"{item.key} {getattr(item.operator, 'value', item.operator)} {item.value}"
        )

    return ", ".join(parts)


def retrieving(test, top_k):
    filter = metadata_extractor(query=test.question)
    context = search_product_hybrid(test.question, top_k=top_k, filters=filter)
    nodes = getattr(context, "nodes", context)
    retrieval_metadata = format_metadata_filters(filter)
    return nodes, retrieval_metadata

In [4]:
async def evaluate_all_retrieval(limit=None, top_k=5, markdown_path=None):
    
    tests = product_load_tests()
    selected_tests = tests[:limit] if limit is not None else tests
    rows = []
    question_reports = []

    for index, test in enumerate(selected_tests, start=1):
        nodes, retrieval_metadata = retrieving(test, top_k)
        
        metrics = evaluate_retrieval(test, nodes, k=top_k)
        
        result = await context_relevance(ContextRelevanceWrapper(question=test.question, contexts=[get_node_text(item) for item in nodes]))
        metrics["context_relevance"] = result
        rows.append({**metrics, "retrieval_metadata": retrieval_metadata})
        filtered_metrics = {
            key: value
            for key, value in metrics.items()
            if key.startswith("precision") or key.startswith("recall") or key == "mrr" or key.startswith("ndcg") or key == "context_relevance"
        }
        doc_rows = build_retrieval_doc_rows(nodes, test.relevant_docs)
        display_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            metrics=filtered_metrics,
            text_blocks=[("Retrieval metadata", retrieval_metadata)],
            doc_rows=doc_rows,
            index=index,
            total=len(selected_tests),
            metric_columns=4,
        )
        question_report = build_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            retrieved_docs=metrics["retrieved_docs"],
            doc_rows=doc_rows,
            extra_fields={"metrics": filtered_metrics, "retrieval_metadata": retrieval_metadata},
            index=index,
            total=len(selected_tests),
        )
        question_reports.append(question_report)

    
    report_df = pd.DataFrame(rows)
    metric_columns = [column for column in report_df.columns if column.startswith(("precision", "recall", "ndcg")) or column in {"mrr", "context_relevance"}]
    summary_df = report_df[metric_columns].mean().to_frame(name="mean").T.round(3)

    display(HTML("<h2 style='margin:32px 0 12px;color:#24292f;'>Final metrics</h2>"))
    display(summary_df)

    if markdown_path is not None:
        saved_path = export_retrieval_report_to_markdown(
            report_df,
            question_reports,
            Path(markdown_path),
            notebook_path="notebooks/product/09_product_metadata_retrieval.ipynb",
        )
        display(HTML(f"<p style='margin:12px 0 0;color:#1a7f37;'>Markdown report saved to: <code>{escape(str(saved_path))}</code></p>"))

    return report_df

In [5]:
await evaluate_all_retrieval(markdown_path="../../reports/product_metadata_retrieving_evaluation.md", top_k=30, limit=50)

,rank,hit,section_id,title,score,metadata,preview
0,1,yes,16950080,-,1.0000,"brand=Mitera, color=Grey, price=5999.0",Product: Mitera Grey Embellished Sequinned Sem...
1,2,no,17413544,-,0.3415,"brand=Mitera, color=Grey, category=Saree, occa...",Product: Mitera Women Grey & Black Premium Cru...
2,3,no,19276886,-,0.1619,"brand=Mitera, color=Grey, category=Saree, occa...",Product: Mitera Grey Floral Embroidered Net Fu...
3,4,no,16615812,-,0.1325,"brand=Mitera, color=Grey, category=Saree, occa...",Product: Mitera Floral Saree With Embroidered ...
4,5,no,18302978,-,0.0691,"brand=Mitera, color=Grey, category=Banarasi, o...",Product: Mitera Grey & Silver-Toned Floral Zar...
5,6,no,16587914,-,0.0554,"brand=Mitera, color=Grey, category=Banarasi, o...",Product: Mitera Grey & Gold-Toned Woven Design...


,rank,hit,section_id,title,score,metadata,preview
0,1,no,16279046,-,0.7403,"brand=20Dresses, color=Blue, category=Regular ...",Product: 20Dresses Women Blue Trousers Brand: ...
1,2,no,14220194,-,0.6205,"brand=DressBerry, color=Burgundy, category=Reg...",Product: DressBerry Women Burgundy Trousers Br...
2,3,no,13523706,-,0.5000,"brand=Varanga, color=Black, occasion=Daily, pr...",Product: Varanga Women Black & White Geometric...
3,4,no,13647622,-,0.4860,"brand=Varanga, color=Pink, occasion=Daily, pri...",Product: Varanga Women Pink & White Leheriya P...
4,5,no,13769372,-,0.4852,"brand=Varanga, color=Black, occasion=Daily, pr...",Product: Varanga Classic Black and White Bandh...
5,6,no,11421530,-,0.4290,"brand=Ginni Arora Label, color=Red, category=R...",Product: Ginni Arora Label Women Red Slim Fit ...
6,7,no,17393956,-,0.4245,"brand=Fabindia, color=Beige, category=Regular ...",Product: Fabindia Women Beige & Red Striped Co...
7,8,no,16044098,-,0.4189,"brand=SASSAFRAS, color=Blue, category=Regular ...",Product: SASSAFRAS Women Blue Ribbed Trousers ...
8,9,no,18770032,-,0.3983,"brand=Trendyol, color=Black, category=Regular ...",Product: Trendyol Women Black High-Rise Trouse...
9,10,no,17378344,-,0.3859,"brand=Saffron Threads, color=Black, category=R...",Product: Saffron Threads Women Black Original ...


,rank,hit,section_id,title,score,metadata,preview
0,1,yes,11166548,-,1.0000,"brand=Athena, color=Burgundy, category=Tailore...",Product: Athena Women Burgundy Solid Tailored ...
1,2,no,10758214,-,0.0849,"brand=Athena, color=Burgundy, category=Pullove...",Product: Athena Women Burgundy Solid Pullover ...
2,3,no,12086086,-,0.0158,"brand=Athena, color=Burgundy, category=Pencil,...",Product: Athena Women Burgundy Solid Pencil Mi...
3,4,no,11634534,-,0.0058,"brand=Athena, color=Burgundy, category=Basic j...",Product: Athena Women Burgundy Solid Basic Jum...
4,5,no,11634536,-,0.0036,"brand=Athena, color=Burgundy, category=Basic j...",Product: Athena Women Burgundy Solid Basic Jum...


,rank,hit,section_id,title,score,metadata,preview
0,1,yes,18921114,-,1.0000,"brand=InWeave, color=Red, category=A-line, occ...",Product: InWeave Women Red Printed A-Line Skir...
1,2,no,17168254,-,0.9076,"brand=InWeave, color=Red, category=A-line, occ...",Product: InWeave Women Red Printed Pure Cotton...
2,3,no,15760840,-,0.3796,"brand=InWeave, color=Red, category=Regular, oc...",Product: InWeave Red & Gold-Toned Regular Crop...
3,4,no,13497800,-,0.1076,"brand=InWeave, color=Red, category=Regular, oc...",Product: InWeave Women Red Striped A-Line Top ...
4,5,no,18922178,-,0.0765,"brand=InWeave, color=Red, occasion=Western, pr...",Product: InWeave Women Red Flared Palazzos Bra...


,rank,hit,section_id,title,score,metadata,preview
0,1,no,10451734,-,0.8545,"brand=WEAVERS VILLA, color=Mustard, occasion=D...",Product: WEAVERS VILLA Women Mustard Yellow & ...
1,2,no,19218994,-,0.6963,"brand=Mitera, color=Mustard, category=Kanjeeva...",Product: Mitera Mustard & Red Woven Design Zar...
2,3,no,18122478,-,0.6744,"brand=PERFECTBLUE, color=Mustard, category=Sar...",Product: PERFECTBLUE Mustard Yellow & Fuchsia ...
3,4,no,17663018,-,0.6605,"brand=Dupatta Bazaar, color=Mustard, occasion=...",Product: Dupatta Bazaar Mustard Yellow Ethnic ...
4,5,no,16719554,-,0.6425,"brand=Banarasi Style, color=Mustard, occasion=...",Product: Banarasi Style Mustard Woven Design D...
5,6,no,11117288,-,0.6263,"brand=Mitera, color=Mustard, category=Kanjeeva...",Product: Mitera Mustard & Pink Silk Blend Wove...
6,7,no,18135092,-,0.5665,"brand=Mitera, color=Mustard, category=Banarasi...",Product: Mitera Mustard Yellow Ethnic Motifs Z...
7,8,no,16921550,-,0.4961,"brand=Safaa, color=Mustard, category=Dress, oc...",Product: Safaa Women Mustard Woven Design Wool...
8,9,no,18141434,-,0.4889,"brand=MANOHARI, color=Mustard, category=Saree,...",Product: MANOHARI Mustard & Gold-Toned Woven D...
9,10,no,14071540,-,0.4309,"brand=DressBerry, color=Mustard, category=Fron...",Product: DressBerry Women Stylish Mustard Self...


...

""


,rank,hit,section_id,title,score,metadata,preview
0,1,yes,16504028,-,0.8484,"brand=ANVI Be Yourself, color=Blue, category=P...",Product: ANVI Be Yourself Women Stunning Blue ...
1,2,no,14641744,-,0.7464,"brand=FabAlley, color=Blue, category=Peplum, o...",Product: FabAlley Blue Puff Sleeve Chambray Pe...
2,3,no,18514120,-,0.7447,"brand=Paralians, color=Turquoise Blue, categor...",Product: Paralians Turquoise Blue Peplum Top B...
3,4,no,11607414,-,0.6649,"brand=Bhama Couture, color=Blue, category=Pepl...",Product: Bhama Couture Women Blue Printed Pepl...
4,5,no,11607374,-,0.6359,"brand=Bhama Couture, color=Blue, category=Pepl...",Product: Bhama Couture Women Blue Printed Pepl...
5,6,no,18805426,-,0.5692,"brand=BoStreet, color=Blue, category=Peplum, o...",Product: BoStreet Blue Checked Peplum Top Bran...
6,7,no,17445656,-,0.5165,"brand=Vishudh, color=Blue, category=Peplum, oc...",Product: Vishudh Blue Floral Print Peplum Top ...
7,8,no,17872080,-,0.5146,"brand=Taavi, color=Blue, category=Peplum, occa...",Product: Taavi Women Indigo & White Indigo Pri...
8,9,no,8715089,-,0.5028,"brand=plusS, color=Blue, category=Peplum, occa...",Product: plusS Women Blue Printed Peplum Top B...
9,10,no,10783322,-,0.5014,"brand=Sera, color=Navy Blue, category=Peplum, ...",Product: Sera Women Navy Blue Printed Peplum P...


,rank,hit,section_id,title,score,metadata,preview
0,1,yes,16533076,-,1.0,"brand=Madame, color=Peach, category=Pullover, ...",Product: Madame Women Peach-Coloured Self Desi...
1,2,no,18948340,-,0.0,"brand=Madame, color=Peach, category=Styled bac...",Product: Madame Peach-Coloured Cuffed Sleeves ...


,precision,recall,mrr,ndcg,context_relevance
mean,0.212,0.64,0.516,0.551,0.795


,question,relevant_docs,retrieved_docs,precision,recall,mrr,ndcg,context_relevance,retrieval_metadata
0,"Can you find something similar to ""Mitera Grey...",[16950080],"[16950080, 17413544, 19276886, 16615812, 18302...",0.166667,1.000,1.000000,1.000000,1.00,"brand == Mitera, color == Grey"
1,Show me Regular trousers within a budget of 2000.,"[17817750, 18770002, 18769968, 18769944, 18769...","[16279046, 14220194, 13523706, 13647622, 13769...",0.000000,0.000,0.000000,0.000000,0.75,price <= 2000.0
2,"Can you find something similar to ""Athena Wome...",[11166548],"[11166548, 10758214, 12086086, 11634534, 11634...",0.200000,1.000,1.000000,1.000000,1.00,"brand == Athena, color == Burgundy"
3,"Show me products like ""InWeave Women Red Print...",[18921114],"[18921114, 17168254, 15760840, 13497800, 18922...",0.200000,1.000,1.000000,1.000000,1.00,"brand == InWeave, color == Red"
4,I need product with a Woven design pattern in ...,"[12824928, 18262088]","[10451734, 19218994, 18122478, 17663018, 16719...",0.066667,1.000,0.076923,0.288586,1.00,color == Mustard
5,I'm looking for products from MANGO for everyd...,"[15315768, 15265896, 15265898, 16892568, 15977...",[],0.000000,0.000,0.000000,0.000000,0.00,"brand == MANGO, occasion == Daily, price <= 56..."
6,What do you have from Sweet Dreams in Playsuit...,"[11581420, 11581366]","[11581420, 11581448, 11581366, 12882594]",0.500000,1.000,1.000000,0.919721,1.00,"brand == Sweet Dreams, color == Pink"
7,I'm looking for Kurta set by Vishudh.,"[13536726, 15997054, 18262138, 13119222, 18263...","[13799826, 14860664, 13536726, 7572969, 159970...",0.266667,1.000,0.333333,0.625509,1.00,brand == Vishudh
8,I'm looking for A-line by Ancestry.,"[15407228, 19152780, 14873702]","[14873856, 15407228, 14873702, 19152780, 14873...",0.250000,1.000,0.500000,0.732829,1.00,brand == Ancestry
9,"Can you find something similar to ""FASHOR Wome...",[19140952],"[19140952, 18964098, 19198414, 19324632]",0.250000,1.000,1.000000,1.000000,1.00,"brand == FASHOR, color == Blue"
